In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from sklearn.model_selection import train_test_split

from customer_intelligence_churn_prediction_system.data.load_data import (
    load_raw_data
)

from customer_intelligence_churn_prediction_system.preprocessing.clean_data import (
    clean_customer_data
)

from customer_intelligence_churn_prediction_system.features.build_features import (
    build_customer_features
)
from customer_intelligence_churn_prediction_system.models.evaluate_models import (
    evaluate_model
)

In [ ]:

df = load_raw_data()

cleaned_df = clean_customer_data(df)

featured_df = build_customer_features(cleaned_df)

print("Featured dataset shape:", featured_df.shape)

Featured dataset shape: (7043, 25)


In [ ]:

X = featured_df.drop(columns=["customerID", "Churn"])

y = featured_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 23)
y shape: (7043,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (5634, 23)
Testing shape: (1409, 23)


In [5]:
from customer_intelligence_churn_prediction_system.preprocessing.preprocess import (
    create_preprocessor
)

from customer_intelligence_churn_prediction_system.models.train_models import (
    build_logistic_regression_pipeline
)

preprocessor = create_preprocessor(X_train)

logistic_model = build_logistic_regression_pipeline(preprocessor)

logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed!")

Logistic Regression training completed!


In [6]:
logistic_metrics = evaluate_model(
    logistic_model,
    X_test,
    y_test
)

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[932 103]
 [179 195]]


In [7]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_random_forest_pipeline
)

rf_preprocessor = create_preprocessor(X_train)

random_forest_model = build_random_forest_pipeline(
    rf_preprocessor
)

random_forest_model.fit(X_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [8]:
rf_metrics = evaluate_model(
    random_forest_model,
    X_test,
    y_test
)

print("\nRandom Forest Metrics:")

for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.61      0.48      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409

Confusion Matrix:
[[920 115]
 [193 181]]

Random Forest Metrics:
Accuracy: 0.7814
Precision: 0.6115
Recall: 0.4840
F1 Score: 0.5403
ROC-AUC: 0.8225


In [10]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_xgboost_pipeline
)

xgb_preprocessor = create_preprocessor(X_train)

xgboost_model = build_xgboost_pipeline(xgb_preprocessor)

xgboost_model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [11]:
xgb_metrics = evaluate_model(
    xgboost_model,
    X_test,
    y_test
)

print("\nXGBoost Metrics:")

for metric, value in xgb_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.51      0.57       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[934 101]
 [183 191]]

XGBoost Metrics:
Accuracy: 0.7984
Precision: 0.6541
Recall: 0.5107
F1 Score: 0.5736
ROC-AUC: 0.8425


In [13]:
from customer_intelligence_churn_prediction_system.models.train_models import (
    build_lightgbm_pipeline
)

lgbm_preprocessor = create_preprocessor(X_train)
lightgbm_model = build_lightgbm_pipeline(lgbm_preprocessor)

lightgbm_model.fit(X_train, y_train)

print("LightGBM training completed!")

LightGBM training completed!


In [17]:
lgbm_metrics = evaluate_model(
    lightgbm_model,
    X_test,
    y_test
)

print("\nLightGBM Metrics:")

for metric, value in lgbm_metrics.items():
    print(f"{metric}: {value:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409

Confusion Matrix:
[[920 115]
 [176 198]]

LightGBM Metrics:
Accuracy: 0.7935
Precision: 0.6326
Recall: 0.5294
F1 Score: 0.5764
ROC-AUC: 0.8334


In [16]:
model_comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **logistic_metrics},
    {"Model": "Random Forest", **rf_metrics},
    {"Model": "XGBoost", **xgb_metrics},
    {"Model":"LightGBM", **lgbm_metrics}
])

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.799858,0.654362,0.521390,0.580357,0.842375
1,Random Forest,0.781405,0.611486,0.483957,0.540299,0.822530
2,XGBoost,0.798439,0.654110,0.510695,0.573574,0.842453
3,LightGBM,0.793471,0.632588,0.529412,0.576419,0.833450


## Baseline Model Comparison — Findings

Four baseline classification models were evaluated:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

Key observations:
- Logistic Regression achieved the best overall F1-score (0.5804).
- XGBoost achieved the highest ROC-AUC (0.8425).
- LightGBM achieved the highest churn recall (0.5294).
- Random Forest showed the weakest baseline performance.

Conclusion:
Logistic Regression is currently the strongest overall baseline.
However, no final model has been selected. Further optimization
and evaluation are required.